[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llm-engineering-certified/notebooks/day-02-chat-models-prompts.ipynb#scrollTo=a2b3c4d5)

---
# Day 2 · Chat Models and Prompt Templates
**certified-journeys / llm-engineering-certified** · Day 2 · Prompting

> **Goal for today:** By the end of this notebook you can build reusable `ChatPromptTemplate` pipelines with system and human message slots, inject variables at call-time, create few-shot prompts using `FewShotChatMessagePromptTemplate`, and compare gpt-3.5-turbo vs gpt-4o-mini on quality, cost, and latency.


In [ ]:
%pip install -q langchain langchain-openai langchain-community python-dotenv


## Step 1 · Environment setup and API key

We load `OPENAI_API_KEY` from a `.env` file. A mock key lets you explore the notebook offline — replace it with a real key to make live API calls.


In [ ]:
import os
import time
import json
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "sk-placeholder-replace-with-real-key"
    print("[INFO] Using placeholder API key — set OPENAI_API_KEY in your .env file.")
else:
    print("[OK] OPENAI_API_KEY loaded from environment.")


### What just happened?
- `load_dotenv()` silently succeeds even if `.env` doesn't exist — no crash.
- **Never hardcode keys** in notebooks — they end up in git history.
- The placeholder key lets all imports and template construction run without hitting the API.
- In Colab, use `Secrets` (🔑 icon in sidebar) to inject `OPENAI_API_KEY` into `os.environ` automatically.


## Step 2 · Why prompt templates? The problem with f-strings

Hardcoding prompts as f-strings works until you need to:
- Swap models without changing prompt logic
- Validate that all required variables are provided
- Log, version, or share prompt structure across a team
- Add few-shot examples cleanly

`ChatPromptTemplate` solves all of this. It separates **structure** (the template) from **data** (the variables).

| Approach | Reusable | Type-safe | Composable | Few-shot support |
|---|---|---|---|---|
| f-string | ✗ | ✗ | ✗ | ✗ |
| `ChatPromptTemplate` | ✓ | ✓ | ✓ | ✓ |

**Official docs:** https://python.langchain.com/docs/concepts/prompt_templates/


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Build a template with two slots: {language} and {code_snippet}.
# SystemMessage sets the persona; HumanMessage is the user's question.
template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {language} developer. Be concise — answer in 3 sentences max."),
    ("human", "Explain what this code does:\n\n```{language}\n{code_snippet}\n```"),
])

# Inspect the template structure before filling in variables.
print("Input variables:", template.input_variables)
print("Number of messages in template:", len(template.messages))
print()

# .format_messages() renders the template into a concrete list of messages.
filled_messages = template.format_messages(
    language="Python",
    code_snippet="result = [x**2 for x in range(10) if x % 2 == 0]",
)

for msg in filled_messages:
    print(f"{type(msg).__name__:20s}: {msg.content[:80]}")


### What just happened?
- `ChatPromptTemplate.from_messages()` accepts `(role, template_string)` tuples — no need to import message classes manually.
- **`input_variables`** lists every `{placeholder}` found in the template — LangChain validates that all are provided at call time.
- `.format_messages()` returns a list of `SystemMessage` / `HumanMessage` objects, ready to pass to `.invoke()`.
- Reuse the same template with different variable values — no copy-pasting prompt strings.


## Step 3 · Invoke the template through a model (LCEL pipe)

LangChain's Expression Language (LCEL) lets you chain a prompt template directly to a model using the `|` operator:

```
chain = prompt | llm
chain.invoke({variables}) → AIMessage
```

The template formats the messages, the model receives them, and the result is an `AIMessage`. No intermediate variable needed.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage

# Instantiate model — temperature=0 for deterministic, reproducible outputs.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=256)

# LCEL chain: template → model.
chain = template | llm

# --- MOCK BLOCK: remove when you have a real key ---
response = AIMessage(
    content="This list comprehension squares every even number from 0–9, producing [0, 4, 16, 36, 64]. It uses a filter `if x % 2 == 0` to skip odd values before squaring. The result is a list of five perfect squares.",
    usage_metadata={"input_tokens": 68, "output_tokens": 45, "total_tokens": 113},
    response_metadata={"model_name": "gpt-4o-mini", "finish_reason": "stop"},
    id="chatcmpl-mock-day02-01",
)
# --- end mock block ---

# Uncomment to make a real call:
# response = chain.invoke({"language": "Python", "code_snippet": "result = [x**2 for x in range(10) if x % 2 == 0]"})

print("Model response:")
print(response.content)
print()
print("Tokens used:", response.usage_metadata)


### What just happened?
- `template | llm` creates a `RunnableSequence` — calling `.invoke()` on it passes data through each stage automatically.
- **Variable dict** is passed to `.invoke()`, not to the template directly — the chain handles routing.
- The output is a standard `AIMessage` with the same `.content`, `.usage_metadata`, and `.response_metadata` as a direct `.invoke()` call.
- LCEL chains are **lazy** — `.invoke()` triggers execution; the chain object itself does no work at definition time.


## Step 4 · Few-shot prompting with `FewShotChatMessagePromptTemplate`

Few-shot prompting gives the model examples of the input→output pattern before asking it to handle a new input. This dramatically improves consistency for tasks like:
- Structured data extraction
- Classification with specific label names
- Output format enforcement

**Official docs:** https://python.langchain.com/docs/how_to/few_shot_examples_chat/

`FewShotChatMessagePromptTemplate` inserts the examples as Human/AI message pairs in the conversation history.


In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# --- Define the few-shot examples ---
# Each example has the same keys as the example_prompt template.
examples = [
    {
        "input": "The server returned a 500 error after the deployment.",
        "output": "Severity: HIGH | Category: Infrastructure | Action: rollback deployment",
    },
    {
        "input": "Login page CSS is slightly misaligned on mobile Safari.",
        "output": "Severity: LOW | Category: UI | Action: create ticket for next sprint",
    },
    {
        "input": "Database connection pool is exhausted under peak load.",
        "output": "Severity: HIGH | Category: Database | Action: increase pool size immediately",
    },
]

# Template that formats each individual example as a Human → AI pair.
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

# Combine examples into a few-shot block.
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
)

# Wrap in a full ChatPromptTemplate with a system message and the live user input.
full_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an incident triage assistant. Classify incidents using the format shown in the examples."),
    few_shot_prompt,          # inserts the 3 Human/AI example pairs here
    ("human", "{incident}"), # the real user query
])

# Render the full prompt to see what the model will receive.
rendered = full_prompt.format_messages(incident="Memory usage on the API server hit 95%.")
print(f"Total messages sent to model: {len(rendered)}")
print()
for msg in rendered:
    role = type(msg).__name__.replace("Message", "")
    print(f"[{role:6s}] {msg.content[:90]}")


### What just happened?
- `FewShotChatMessagePromptTemplate` expands the `examples` list into alternating `HumanMessage` / `AIMessage` pairs — giving the model a demonstration before the real query.
- **Three examples → 6 extra messages** (3 human + 3 AI), plus the system and final human messages = 8 total messages.
- The model infers the output format from the examples without needing an explicit format description.
- More examples → better format adherence, but higher token cost — start with 3–5 and tune from there.


In [ ]:
# Invoke the few-shot chain.
few_shot_chain = full_prompt | llm

# --- MOCK BLOCK ---
few_shot_response = AIMessage(
    content="Severity: HIGH | Category: Infrastructure | Action: investigate memory leak and restart service if needed",
    usage_metadata={"input_tokens": 190, "output_tokens": 22, "total_tokens": 212},
    response_metadata={"model_name": "gpt-4o-mini", "finish_reason": "stop"},
    id="chatcmpl-mock-day02-02",
)
# --- end mock block ---

# Uncomment to run live:
# few_shot_response = few_shot_chain.invoke({"incident": "Memory usage on the API server hit 95%."})

print("Few-shot triage result:")
print(few_shot_response.content)
print()
print(f"Input tokens: {few_shot_response.usage_metadata['input_tokens']} "
      f"(examples add ~{few_shot_response.usage_metadata['input_tokens'] - 68} tokens vs zero-shot)")


### What just happened?
- The model followed the `Severity | Category | Action` format exactly — no format instructions needed in the system message.
- **Input tokens jumped** because all three example pairs were included in the prompt — always measure this trade-off.
- Few-shot works best when zero-shot output format is inconsistent; if the model already follows format reliably, few-shot is wasted tokens.
- For dynamic example selection (e.g., picking the 3 most similar examples from a larger pool), use `SemanticSimilarityExampleSelector`.


## Step 5 · Compare gpt-3.5-turbo vs gpt-4o-mini — quality, cost, and latency

Choosing a model is an engineering decision, not just a product one. Key variables:

| Model | Input price (per 1M tokens) | Output price | Context window | Best for |
|---|---|---|---|---|
| `gpt-3.5-turbo` | $0.50 | $1.50 | 16K | High-volume, simple tasks |
| `gpt-4o-mini` | $0.15 | $0.60 | 128K | Everyday experimentation, instruction-following |
| `gpt-4o` | $5.00 | $15.00 | 128K | Complex reasoning, tool use |

We'll run the same prompt on both budget models and compare output quality, token usage, and latency.


In [ ]:
# Shared evaluation prompt — ask both models the same question.
eval_template = ChatPromptTemplate.from_messages([
    ("system", "You are a senior software engineer. Be precise and concise."),
    ("human", "What are the three main risks of using a mutable default argument in Python functions? Give a brief code example illustrating the problem."),
])

# Define both models.
model_35 = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, max_tokens=300)
model_4o_mini = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=300)

# --- MOCK BLOCK: simulates two API calls with timing ---
results = {
    "gpt-3.5-turbo": {
        "content": """1. Shared state across calls: the default is created once at function definition, not per call.\n2. Silent bugs: callers don't know the list/dict accumulates state between invocations.\n3. Hard to test: test order affects results.\n\nExample:\n```python\ndef append_to(item, lst=[]):\n    lst.append(item)\n    return lst\n\nprint(append_to(1))  # [1]\nprint(append_to(2))  # [1, 2] — surprise!\n```""",
        "input_tokens": 52, "output_tokens": 98, "latency_s": 0.82,
    },
    "gpt-4o-mini": {
        "content": """Using a mutable default argument (e.g., `def f(x, data=[])`) has three key risks:\n\n1. **Persistent state**: The default object is shared across all calls — mutations accumulate.\n2. **Unpredictable behavior**: Later callers see side effects from earlier callers.\n3. **Thread safety**: Concurrent calls may corrupt shared state.\n\n```python\ndef add_item(item, lst=[]):  # DON'T DO THIS\n    lst.append(item)\n    return lst\n\nadd_item('a')  # ['a']\nadd_item('b')  # ['a', 'b']  ← list grew between calls\n```\n\nFix: use `None` as default and initialise inside the function.""",
        "input_tokens": 52, "output_tokens": 138, "latency_s": 1.04,
    },
}
# --- end mock block ---

# Uncomment for live comparison:
# results = {}
# for name, model in [("gpt-3.5-turbo", model_35), ("gpt-4o-mini", model_4o_mini)]:
#     chain = eval_template | model
#     t0 = time.time()
#     resp = chain.invoke({})
#     results[name] = {
#         "content": resp.content,
#         **resp.usage_metadata,
#         "latency_s": round(time.time() - t0, 2),
#     }

for model_name, data in results.items():
    print(f"=== {model_name} ===")
    print(data["content"])
    print()


### What just happened?
- Both models answered correctly, but `gpt-4o-mini` added **bold labels** and a **fix suggestion** — richer output for similar cost.
- `gpt-3.5-turbo` was faster but produced fewer output tokens — fine for simple retrieval tasks, weaker for nuanced explanation.
- Both chains used exactly the same `ChatPromptTemplate` — only the model changed.
- **Template reuse** is the key benefit: update the prompt once, both models benefit.


## Step 6 · Log temperature, model name, and latency for each call

In production you need a structured call log for:
- Cost attribution (which feature called which model?)
- Latency monitoring (are response times within SLA?)
- Quality tracking (correlate temperature with output variance)

We'll build a lightweight `CallLogger` that wraps a chain and records metadata per call.


In [ ]:
import time
from dataclasses import dataclass, field, asdict
from typing import Any

PRICING = {
    "gpt-3.5-turbo": {"input": 0.50 / 1_000_000, "output": 1.50 / 1_000_000},
    "gpt-4o-mini":   {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
    "gpt-4o":        {"input": 5.00 / 1_000_000, "output": 15.00 / 1_000_000},
}

@dataclass
class CallRecord:
    model: str
    temperature: float
    input_tokens: int
    output_tokens: int
    latency_s: float
    cost_usd: float
    content_preview: str  # first 80 chars of response


def logged_invoke(chain, variables: dict, model_name: str, temperature: float) -> CallRecord:
    """Invoke a chain and return a structured CallRecord with timing and cost."""
    t0 = time.time()
    # In mock mode we use pre-computed results; in live mode: response = chain.invoke(variables)
    mock_data = results.get(model_name, {})
    latency = mock_data.get("latency_s", round(time.time() - t0, 3))
    input_tokens = mock_data.get("input_tokens", 0)
    output_tokens = mock_data.get("output_tokens", 0)
    content = mock_data.get("content", "")

    pricing = PRICING.get(model_name, {"input": 0, "output": 0})
    cost = round(
        input_tokens * pricing["input"] + output_tokens * pricing["output"], 8
    )

    return CallRecord(
        model=model_name,
        temperature=temperature,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        latency_s=latency,
        cost_usd=cost,
        content_preview=content[:80],
    )


# Run both models through the logger.
log = []
for model_name, model_obj in [("gpt-3.5-turbo", model_35), ("gpt-4o-mini", model_4o_mini)]:
    chain = eval_template | model_obj
    record = logged_invoke(chain, {}, model_name, temperature=0)
    log.append(record)

# Print a comparison table.
print(f"{'Model':<20} {'Temp':>5} {'In tok':>7} {'Out tok':>8} {'Latency':>9} {'Cost USD':>12}")
print("-" * 70)
for r in log:
    print(f"{r.model:<20} {r.temperature:>5.1f} {r.input_tokens:>7} {r.output_tokens:>8} {r.latency_s:>8.2f}s {r.cost_usd:>12.8f}")


### What just happened?
- `CallRecord` is a typed dataclass — easy to serialise to JSON or write to a database.
- **`gpt-4o-mini` is cheaper per token than `gpt-3.5-turbo`** despite being a newer, more capable model — OpenAI repriced it aggressively.
- Logging `temperature` alongside the result lets you retrospectively correlate output variance with randomness settings.
- For production logging, use LangSmith (free tier available) or write `asdict(record)` to a structured log sink.


In [ ]:
# Challenge: Multi-model prompt experiment
# ─────────────────────────────────────────
# Build a ChatPromptTemplate for a sentiment classification task:
#   System: "Classify the sentiment of the review. Reply with exactly one word: Positive, Negative, or Neutral."
#   Human:  "{review}"
#
# Then:
#   1. Create a FewShotChatMessagePromptTemplate with at least 2 examples.
#   2. Run the few-shot chain on gpt-3.5-turbo AND gpt-4o-mini for 3 different reviews.
#   3. Log results using CallRecord (or a similar structure).
#   4. Print a summary table showing model, review snippet, predicted label, and cost.
#
# Hint: you can reuse the logged_invoke() function from Step 6.

# Your solution here
reviews = [
    "Absolutely love this product — fast shipping and exactly as described!",
    "Stopped working after two days. Very disappointed.",
    "It's okay, nothing special but does the job.",
]

# TODO: build few_shot_sentiment_template
# TODO: run on both models for each review
# TODO: print summary table


---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| `ChatPromptTemplate` | Separates prompt structure from data — use `.from_messages()` with `(role, template)` tuples |
| `{variable}` slots | Declared in template strings; `.input_variables` lists them; `.format_messages()` fills them |
| LCEL pipe `template | llm` | Creates a `RunnableSequence` — `.invoke(vars)` runs the full chain |
| `FewShotChatMessagePromptTemplate` | Inserts example Human/AI pairs before the live query — improves format adherence |
| `example_prompt` | Per-example template used to render each few-shot pair |
| gpt-3.5-turbo vs gpt-4o-mini | 4o-mini is cheaper AND has a larger context window — prefer it for everyday use |
| `CallRecord` / logging | Always log model, temperature, tokens, latency, and cost per call |

> **Tip:** Use `model=gpt-4o-mini` for everyday experimentation — it's 10x cheaper than gpt-4o with similar instruction-following quality.

---
## What's next
**Day 3** → Output Parsers and Structured Outputs — learn to extract clean strings, typed Pydantic objects, and schema-free JSON from model responses.

Mark Day 2 complete in your [tracker](../index.html).
